# 00 — nba_api Smoke Test

Phase 0, Day 2. Pull one game per season 2019-20 through 2024-25 via `nba_api`'s `playbyplayv3` endpoint. Verify schema parity across seasons; log timing and rate-limit behavior. Save raw JSONs to `data/raw/`.

**Result:** 6/6 seasons pulled cleanly. Schema **identical** across all 6 (24 columns). Median round-trip ~0.5s; one transient 22s outlier consistent with a stats.nba.com hiccup, not a hard rate limit. PBP v3 gives us everything we need for game-state features: `clock`, `period`, `scoreHome`/`scoreAway`, `teamId`, `shotResult`, `shotValue`, `actionType`/`subType`.

The fetching code lives in `src/data/pull_pbp.py`; the orchestration is `scripts/smoke_test_nba_api.py`. Re-running this notebook re-uses the cached JSONs in `data/raw/` and does not re-hit the API.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from src.data.pull_pbp import DATA_RAW, fetch_pbp, first_game_id, season_label

## Summary of the smoke run

Loaded from the CSV the script writes after running.

In [ ]:
summary = pd.read_csv(DATA_RAW / 'smoke_test_summary.csv')
summary

## Inspect one game's PBP frame

Use the most recent season as the working example.

In [ ]:
res = fetch_pbp(game_id=first_game_id(2024), season=season_label(2024))
print(f'status={res.status}  elapsed={res.elapsed_s:.2f}s  n_events={len(res.df)}')
print('columns:', list(res.df.columns))
res.df.head(10)

## Sanity: scoring monotonicity

`scoreHome` and `scoreAway` should be monotonically non-decreasing within a game.

In [ ]:
df = res.df.copy()
df['scoreHome'] = pd.to_numeric(df['scoreHome'], errors='coerce').ffill().fillna(0).astype(int)
df['scoreAway'] = pd.to_numeric(df['scoreAway'], errors='coerce').ffill().fillna(0).astype(int)
home_ok = (df['scoreHome'].diff().fillna(0) >= 0).all()
away_ok = (df['scoreAway'].diff().fillna(0) >= 0).all()
print(f'home monotonic: {home_ok}   away monotonic: {away_ok}')
print(f'final score: home {df.scoreHome.iloc[-1]} - away {df.scoreAway.iloc[-1]}')

## Next steps (Day 3)

- Complete the Odds API free-tier audit (Task #4): sign up, read docs, email support, run one test query.
- Survey alternate odds sources (Task #5) and write `data_sources.md`.
- GO/NO-GO call on live-odds source (Task #6).
- All before any test-set odds get pulled — the trailing-team hypothesis is pre-registered in `DECISIONS.md`.